# Performance Internals — 03: Memory Management

## What you will learn
Memory management is the most common source of Spark job failures
(`OutOfMemoryError`, `GC overhead limit exceeded`, shuffle spill).
Understanding how Spark allocates memory is essential for tuning production jobs.

In this notebook you will:
1. Understand the complete executor memory model
2. See how Spark splits memory between execution and storage
3. Observe shuffle spill and learn how to prevent it
4. Understand off-heap memory and when to use it
5. Tune GC settings for long-running jobs
6. Profile memory usage in practice

## The Executor Memory Model

```
JVM Process Memory (spark.executor.memory = 2g)
├── Reserved Memory          = 300 MB  (fixed, for Spark internals)
├── User Memory              = (1 - spark.memory.fraction) × (heap - 300MB)
│     └── Your data structures, UDF state, broadcast variables
└── Spark Memory Pool        = spark.memory.fraction × (heap - 300MB)  [default: 0.6]
      ├── Storage Memory     = spark.memory.storageFraction × pool  [default: 0.5]
      │     └── Cached RDDs/DataFrames, broadcast vars
      └── Execution Memory   = rest of pool
            └── Shuffle, sort, aggregation, join hash tables

Off-Heap (spark.memory.offHeap.enabled):
└── Separate from JVM heap — not subject to GC
      └── Used by Tungsten for sort/shuffle buffers

Container Memory (spark.executor.memoryOverhead):
└── Native memory outside JVM (Python worker, NIO buffers)
```

Key insight: **Execution and Storage share the Spark Memory Pool.**
When execution needs more memory, it can evict storage (cached data).
When storage needs more, it can claim unused execution memory.
This is called **unified memory management** (introduced in Spark 1.6).


In [ ]:
import os, time, random, datetime, pathlib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'

spark = (
    SparkSession.builder
    .appName('memory-management')
    .master(MASTER)
    .config('spark.executor.memory',            '2g')
    .config('spark.executor.memoryOverhead',    '512m')
    .config('spark.driver.memory',              '1g')
    .config('spark.executor.cores',             '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
# Re-suppress noisy loggers after setLogLevel resets them (Spark 4.x / log4j2)
_jvm = spark.sparkContext._jvm
_ctx = _jvm.org.apache.logging.log4j.LogManager.getContext(False)
_cfg = _ctx.getConfiguration()
for _logger_name in [
    'org.apache.iceberg.hadoop.HadoopTableOperations',
    'org.apache.hadoop.util.NativeCodeLoader',
]:
    _lc = _jvm.org.apache.logging.log4j.core.config.LoggerConfig(_logger_name,
              _jvm.org.apache.logging.log4j.Level.ERROR, False)
    _cfg.addLogger(_logger_name, _lc)
_ctx.updateLoggers()
print(f"Spark {spark.version}")

## Step 1 — Inspect Current Memory Configuration

Let's see how memory is configured and calculate the actual allocation.


In [ ]:
def get_conf(key, default="(not set)"):
    try:    return spark.conf.get(key)
    except: return default

# Core memory settings
mem_settings = {
    "spark.executor.memory":              get_conf("spark.executor.memory"),
    "spark.executor.memoryOverhead":      get_conf("spark.executor.memoryOverhead"),
    "spark.driver.memory":                get_conf("spark.driver.memory"),
    "spark.memory.fraction":              get_conf("spark.memory.fraction", "0.6"),
    "spark.memory.storageFraction":       get_conf("spark.memory.storageFraction", "0.5"),
    "spark.memory.offHeap.enabled":       get_conf("spark.memory.offHeap.enabled", "false"),
    "spark.memory.offHeap.size":          get_conf("spark.memory.offHeap.size", "0"),
    "spark.shuffle.spill.compress":       get_conf("spark.shuffle.spill.compress", "true"),
    "spark.sql.shuffle.partitions":       get_conf("spark.sql.shuffle.partitions", "200"),
}

print("Memory Configuration:")
print("-" * 60)
for k, v in mem_settings.items():
    print(f"  {k.split('.')[-1]:<35} {v}")

# Calculate actual allocations
heap_mb       = 2048  # spark.executor.memory = 2g
reserved_mb   = 300
usable_mb     = heap_mb - reserved_mb
fraction      = float(get_conf("spark.memory.fraction", "0.6"))
storage_frac  = float(get_conf("spark.memory.storageFraction", "0.5"))

spark_pool_mb    = usable_mb * fraction
user_mb          = usable_mb * (1 - fraction)
storage_mb       = spark_pool_mb * storage_frac
execution_mb     = spark_pool_mb * (1 - storage_frac)

print()
print("Calculated memory layout (executor heap = 2 GB):")
print(f"  Reserved memory      : {reserved_mb:>6} MB  (fixed Spark internals)")
print(f"  User memory          : {user_mb:>6.0f} MB  (your code, UDF state)")
print(f"  Spark pool total     : {spark_pool_mb:>6.0f} MB")
print(f"    Storage memory     : {storage_mb:>6.0f} MB  (cache, broadcasts)")
print(f"    Execution memory   : {execution_mb:>6.0f} MB  (shuffle, sort, agg)")
print()
print("Note: Storage and Execution share the pool dynamically.")
print("If execution needs more it can evict cached data (not the reverse).")

## Step 2 — Storage Memory: Caching and Persistence Levels

Spark's `cache()` uses Storage Memory. When the cache fills up,
Spark evicts the least-recently-used partitions.

Understanding persistence levels:
| Level | Where | Recompute on eviction | Best for |
|---|---|---|---|
| `MEMORY_ONLY` | JVM heap | Yes | Small DataFrames that fit in memory |
| `MEMORY_AND_DISK` | Heap, spill to disk | No (read disk) | Default for production |
| `MEMORY_ONLY_SER` | Heap, serialized | Yes | Reduce memory footprint 2-5x |
| `MEMORY_AND_DISK_SER` | Heap + disk, serialized | No | Large DataFrames |
| `DISK_ONLY` | Disk only | No | Rarely fits in memory |
| `OFF_HEAP` | Off-heap | Yes | Avoid GC pressure |


In [ ]:
from pyspark import StorageLevel

random.seed(42)
N = 500_000

df = spark.range(N).select(
    F.col("id"),
    (F.col("id") % 100).alias("group"),
    F.rand(42).alias("value"),
    F.concat(F.lit("user_"), F.col("id").cast("string")).alias("name"),
)

# Compare cache sizes at different storage levels
print("Caching the same DataFrame at different storage levels:")
print("-" * 65)

for level_name, level in [
    ("MEMORY_ONLY",          StorageLevel.MEMORY_ONLY),
    ("MEMORY_ONLY_SER",      StorageLevel.MEMORY_ONLY_2),   # uses kryo-style
    ("MEMORY_AND_DISK",      StorageLevel.MEMORY_AND_DISK),
]:
    df.persist(level)
    df.count()   # materialize

    # Get storage info from SparkContext
    rdd_info = spark.sparkContext._jsc.sc().getRDDStorageInfo()
    cached_mb = sum(
        info.memSize() for info in rdd_info
    ) / 1024 / 1024

    print(f"  {level_name:<25} cached: {cached_mb:>6.1f} MB")
    df.unpersist()

print()
print("MEMORY_ONLY_SER uses ~2x less memory than MEMORY_ONLY")
print("at the cost of CPU time for serialization/deserialization.")

## Step 3 — Execution Memory: Shuffle Spill

When a shuffle (sort, groupBy, join) needs more memory than available,
Spark **spills** data to disk. Spill is not a crash — it's a safety valve.
But it is **very expensive** because it involves disk I/O.

We deliberately create a spill situation and observe it.


In [ ]:
# Create a dataset that requires significant shuffle memory
# Many distinct keys × complex aggregation = high shuffle memory demand
random.seed(42)

N = 2_000_000
large_df = spark.range(N).select(
    F.col("id"),
    (F.col("id") % 500000).alias("high_card_key"),  # 500k distinct keys
    F.rand(42).alias("val1"),
    F.rand(43).alias("val2"),
    F.rand(44).alias("val3"),
)

# Enable spill metrics
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # force sort-merge

print("Running aggregation that may trigger shuffle spill...")
print("Watch Spark UI Stages tab for 'Shuffle Spill (Memory)' and 'Shuffle Spill (Disk)'")
print()

t0 = time.time()
result = (
    large_df
    .groupBy("high_card_key")
    .agg(
        F.sum("val1").alias("s1"),
        F.sum("val2").alias("s2"),
        F.avg("val3").alias("a3"),
        F.count("*").alias("cnt")
    )
    .agg(F.sum("s1"), F.avg("a3"), F.count("*"))
    .collect()
)
elapsed = time.time() - t0
print(f"Aggregation completed in {elapsed:.2f}s")
print(f"Result: {result[0]}")
print()
print("If spill occurred, you will see in Spark UI:")
print("  Stages tab → find the shuffle stage → 'Spill (Memory)' and 'Spill (Disk)' columns")

In [ ]:
# Monitor spill via accumulators (available after job completes)
# Spark exposes spill metrics through the REST API at port 4040

import urllib.request, json as pyjson

try:
    url = "http://spark-master:4040/api/v1/applications"
    with urllib.request.urlopen(url, timeout=3) as r:
        apps = pyjson.loads(r.read())
    if apps:
        app_id = apps[0]["id"]
        url2 = f"http://spark-master:4040/api/v1/applications/{app_id}/stages"
        with urllib.request.urlopen(url2, timeout=3) as r:
            stages = pyjson.loads(r.read())

        print("Recent stages with spill metrics:")
        print(f"{'Stage':<8} {'Tasks':<8} {'Spill(mem) MB':<16} {'Spill(disk) MB':<16} {'Duration ms':<12}")
        print("-" * 65)
        for s in stages[:5]:
            spill_mem  = s.get("memoryBytesSpilled", 0) / 1024 / 1024
            spill_disk = s.get("diskBytesSpilled",   0) / 1024 / 1024
            print(f"  {s['stageId']:<6} {s['numTasks']:<8} {spill_mem:>12.1f} MB {spill_disk:>12.1f} MB {s['executorRunTime']:<12}")
except Exception as e:
    print(f"Spark UI API not reachable: {e}")
    print("Check http://localhost:4040/stages manually for spill metrics.")

## Step 4 — Preventing Spill: Tuning Strategies

### Strategy 1: Increase shuffle partitions
More partitions = smaller per-partition data = less memory per task.


In [ ]:
# Default: 200 shuffle partitions → each partition may be large
# Increase to reduce per-partition memory pressure

print("Effect of shuffle partition count on memory pressure:")
print()

for n_parts in [50, 200, 500]:
    spark.conf.set("spark.sql.shuffle.partitions", str(n_parts))
    t0 = time.time()
    r = (large_df.groupBy("high_card_key")
                 .agg(F.sum("val1"), F.count("*"))
                 .count())
    elapsed = time.time() - t0
    print(f"  {n_parts:>4} partitions → {elapsed:.2f}s  ({r:,} groups)")

# Reset
spark.conf.set("spark.sql.shuffle.partitions", "200")
print()
print("More partitions = smaller tasks = less spill risk.")
print("But too many partitions = too much overhead (scheduling, small files).")
print("Rule of thumb: aim for 100-200 MB per partition after shuffle.")

### Strategy 2: Use `persist()` to avoid recomputation

If a DataFrame is used multiple times, cache it.
Without caching, Spark recomputes it from scratch each time — wasting memory
because the lineage is re-executed multiple times.


In [ ]:
# WITHOUT cache: df is computed twice
print("Without cache — df computed twice:")
base = spark.range(500_000).select(
    F.col("id"),
    (F.col("id") % 1000).alias("group"),
    F.rand(42).alias("value")
)

t0 = time.time()
count1 = base.filter(F.col("value") > 0.5).count()
count2 = base.groupBy("group").agg(F.sum("value")).count()
t_no_cache = time.time() - t0
print(f"  Time: {t_no_cache:.2f}s  (base DataFrame computed twice)")

# WITH cache: df computed once, reused
print("\nWith cache — df computed once:")
base_cached = base.cache()
base_cached.count()   # trigger caching

t0 = time.time()
count1c = base_cached.filter(F.col("value") > 0.5).count()
count2c = base_cached.groupBy("group").agg(F.sum("value")).count()
t_cached = time.time() - t0
print(f"  Time: {t_cached:.2f}s  (base DataFrame read from cache)")
print(f"  Speedup: {t_no_cache/t_cached:.1f}x")

base_cached.unpersist()

## Step 5 — Off-Heap Memory

Off-heap memory lives **outside the JVM heap** — it is not managed by GC.

**Benefits of off-heap:**
- No GC pauses on this data
- Can be larger than JVM heap limit
- Used by Tungsten for sort/shuffle buffers

**When to enable:**
- Long-running jobs with frequent GC pauses
- Jobs with large sort/shuffle that cause GC pressure
- When `-Xmx` cannot be increased but more memory is available

Gluten/Velox uses its own native memory management which is effectively
off-heap — this is one reason why Gluten reduces GC pressure significantly.


In [ ]:
# Check if off-heap is enabled
off_heap_enabled = get_conf("spark.memory.offHeap.enabled", "false")
off_heap_size    = get_conf("spark.memory.offHeap.size", "0")

print(f"Off-heap enabled : {off_heap_enabled}")
print(f"Off-heap size    : {off_heap_size}")
print()
print("To enable off-heap (set in SparkSession builder or spark-defaults.conf):")
print("  .config('spark.memory.offHeap.enabled', 'true')")
print("  .config('spark.memory.offHeap.size',    '1g')")
print()
print("Note: In our cluster, Gluten/Velox uses native memory (off-JVM-heap)")
print("for all columnar operations. This is one reason Gluten reduces GC pressure")
print("compared to pure JVM Spark — the Velox engine manages its own memory pool.")
print()

# Show GC metrics from last job
try:
    url = f"http://spark-master:4040/api/v1/applications"
    with urllib.request.urlopen(url, timeout=3) as r:
        apps = pyjson.loads(r.read())
    if apps:
        app_id = apps[0]["id"]
        url2 = f"http://spark-master:4040/api/v1/applications/{app_id}/executors"
        with urllib.request.urlopen(url2, timeout=3) as r:
            executors = pyjson.loads(r.read())
        print("GC metrics per executor:")
        print(f"{'Executor':<12} {'GC Time (ms)':>15} {'Peak Mem MB':>14}")
        print("-" * 45)
        for ex in executors:
            if ex["id"] != "driver":
                gc_ms  = ex.get("totalGCTime", 0)
                mem_mb = ex.get("peakMemoryMetrics", {}).get("JVMHeapMemory", 0) / 1024 / 1024
                print(f"  {ex['id']:<10} {gc_ms:>13,} ms {mem_mb:>12.0f} MB")
except Exception as e:
    print(f"(Executor metrics not available via API: {e})")

## Step 6 — Broadcast Variables: Efficient Small Data Sharing

Broadcast variables send data to all executors **once** and cache it there.
Without broadcast, Spark would serialize and send the data with every task.

**When to use:**
- Lookup tables shared across all tasks (e.g., country codes, product categories)
- Small DataFrames used in joins (auto-broadcast if < `autoBroadcastJoinThreshold`)
- Configuration dictionaries used in UDFs


In [ ]:
# Manual broadcast variable
country_map = {
    "US": "United States", "UK": "United Kingdom", "DE": "Germany",
    "FR": "France", "JP": "Japan", "CA": "Canada", "AU": "Australia",
}

# Broadcast once — all executors get a local copy
bc_countries = spark.sparkContext.broadcast(country_map)

@F.udf(returnType=StringType())
def resolve_country(code):
    return bc_countries.value.get(code, "Unknown")

# Use in DataFrame
events = spark.range(100_000).select(
    F.col("id"),
    F.element_at(F.array([F.lit(c) for c in country_map.keys()]),
                 (F.col("id") % 7 + 1).cast("int")).alias("country_code")
)

with_names = events.withColumn("country_name", resolve_country(F.col("country_code")))
with_names.groupBy("country_name").count().orderBy("country_name").show()

# Cleanup — unpersist broadcast when no longer needed
bc_countries.unpersist()
print("Broadcast variable unpersisted — memory freed on all executors.")
print()
print("Key rule: call .unpersist() on broadcasts when done.")
print("Otherwise they accumulate on executors until job ends.")

## Summary: Memory Tuning Checklist

### When you see OutOfMemoryError
1. **Increase `spark.executor.memory`** — but don't use more than 75% of node RAM
2. **Increase `spark.executor.memoryOverhead`** — for Python/native memory (default 384 MB)
3. **Reduce data size** — filter earlier, select fewer columns
4. **Increase shuffle partitions** — `spark.sql.shuffle.partitions` (default 200)
5. **Cache smarter** — use `MEMORY_AND_DISK` not `MEMORY_ONLY` for large data

### When you see excessive GC
1. **Use Kryo serialization** — `spark.serializer = org.apache.spark.serializer.KryoSerializer`
2. **Enable off-heap** — moves shuffle buffers out of GC scope
3. **Use Gluten/Velox** — native engine avoids JVM GC entirely for columnar ops
4. **Reduce object creation** — use SQL functions instead of Python UDFs

### Memory fraction tuning
| Workload | `spark.memory.fraction` | `spark.memory.storageFraction` |
|---|---|---|
| Default | 0.6 | 0.5 |
| Heavy caching | 0.6 | 0.7 |
| Heavy shuffle/join | 0.7 | 0.3 |
| Streaming (stateful) | 0.6 | 0.2 |

**Next:** `04_join_strategies.ipynb` — deep dive into every join type with benchmarks.
